# 5.7 Derinlemesine: Destek Vektör Makineleri

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/05-sklearn/07-support-vector-machines.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: 05.07 Support Vector Machines

Destek vektör makineleri (SVM), hem sınıflandırma hem regresyon için özellikle güçlü ve esnek bir denetimli algoritma sınıfıdır. Bu bölümde SVM'lerin sezgisini ve sınıflandırma problemlerinde kullanımını inceleyeceğiz.

Standart içe aktarmalarla başlayalım:


In [ ]:
# imports_svm.py
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-whitegrid')
from scipy import stats



> **Not**
>

## Destek Vektör Makinelerini Motive Etmek

Bayes sınıflandırması tartışmamızın bir parçası olarak (bkz. 5.5 Naive Bayes), her sınıfın dağılımını tanımlayan basit bir model türünü ve yeni noktalar için olasılıksal etiket belirlemeyi gördük. Bu üretici sınıflandırma örneğiydi; burada ayırt edici sınıflandırmayı ele alacağız: her sınıfı modellemek yerine, sınıfları birbirinden ayıran bir doğru veya eğri (iki boyutta) ya da çok boyutta bir manifold buluruz.

Örnek olarak, iki sınıf noktasının iyi ayrıldığı basit bir sınıflandırma görevini düşünün (aşağıdaki şekil):


In [ ]:
# make_blobs_svm.py
from sklearn.datasets import make_blobs
X, y = make_blobs(n_samples=50, centers=2,
                  random_state=0, cluster_std=0.60)
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='autumn');



Doğrusal ayırt edici bir sınıflandırıcı, iki veri kümesini ayıran düz bir çizgi çizmeye ve böylece bir sınıflandırma modeli oluşturmaya çalışır. Burada gösterilen gibi iki boyutlu veri için bunu elle yapabiliriz. Ancak hemen bir sorun görürüz: iki sınıfı mükemmel ayıran birden fazla olası ayırıcı doğru vardır!


In [ ]:
# linear_separators.py
xfit = np.linspace(-1, 3.5)
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='autumn')
plt.plot([0.6], [2.1], 'x', color='red', markeredgewidth=2, markersize=10)

for m, b in [(1, 0.65), (0.5, 1.6), (-0.2, 2.9)]:
    plt.plot(xfit, m * xfit + b, '-k')

plt.xlim(-1, 3.5);



Bunlar örnekleri mükemmel ayıran üç çok farklı ayırıcıdır. Hangisini seçerseniz, yeni bir veri noktası (örneğin grafikte "X" ile işaretlenen) farklı etiket alır! Görünüşte "sınıflar arasına çizgi çekme" sezgimiz yeterli değil; biraz daha derin düşünmemiz gerekir.

## Destek Vektör Makineleri: Marjı Maksimize Etmek


In [ ]:
# margin_plot.py
xfit = np.linspace(-1, 3.5)
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='autumn')

for m, b, d in [(1, 0.65, 0.33), (0.5, 1.6, 0.55), (-0.2, 2.9, 0.2)]:
    yfit = m * xfit + b
    plt.plot(xfit, yfit, '-k')
    plt.fill_between(xfit, yfit - d, yfit + d, edgecolor='none',
                     color='lightgray', alpha=0.5)

plt.xlim(-1, 3.5);



Bu marjı maksimize eden doğru, optimal model olarak seçilecektir.

### Destek Vektör Makinesi Uydurma


In [ ]:
# svc_fit.py
from sklearn.svm import SVC # "Support vector classifier"
model = SVC(kernel='linear', C=1E10)
model.fit(X, y)



### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      İki blob üzerinde doğrusal SVM uydurun:
          
      from sklearn.datasets import make_blobs
from sklearn.svm import SVC
X, y = make_blobs(n_samples=50, centers=2, random_state=0)
clf = SVC(kernel='linear', C=1e5)
clf.fit(X, y)
print("Destek vektör sayısı:", clf.support_vectors_.shape[0])

Bu veriye gerçek bir uydurmanın sonucuna bakalım: Scikit-Learn'in destek vektör sınıflandırıcısını (SVC) kullanarak bu veride bir SVM modeli eğiteceğiz. Şimdilik doğrusal çekirdek ve C parametresini çok büyük bir değere ayarlayacağız (anlamlarını kısa süre içinde tartışacağız):


In [ ]:
# plot_svc_decision_function.py
def plot_svc_decision_function(model, ax=None, plot_support=True):
    """Plot the decision function for a 2D SVC"""
    if ax is None:
        ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    # create grid to evaluate model
    x = np.linspace(xlim[0], xlim[1], 30)
    y = np.linspace(ylim[0], ylim[1], 30)
    Y, X = np.meshgrid(y, x)
    xy = np.vstack([X.ravel(), Y.ravel()]).T
    P = model.decision_function(xy).reshape(X.shape)
    
    # plot decision boundary and margins
    ax.contour(X, Y, P, colors='k',
               levels=[-1, 0, 1], alpha=0.5,
               linestyles=['--', '-', '--'])
    
    # plot support vectors
    if plot_support:
        ax.scatter(model.support_vectors_[:, 0],
                   model.support_vectors_[:, 1],
                   s=300, linewidth=1, edgecolors='black',
                   facecolors='none');
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)



In [ ]:
# svc_scatter.py
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='autumn')
plot_svc_decision_function(model);



Bu, iki nokta kümesi arasındaki marjı maksimize eden ayırıcı doğrudur. Birkaç eğitim noktasının marja tam dokunduğunu — aşağıdaki şekilde daire içine alındığını — fark edin. Bu noktalar uydurmanın kritik öğeleridir; destek vektörleri olarak bilinir ve algoritmaya adını verir. Scikit-Learn'de kimlikleri sınıflandırıcının support_vectors_ özniteliğinde saklanır:


In [ ]:
# support_vectors_.py
model.support_vectors_



Bu sınıflandırıcının başarısının anahtarı, uydurma için yalnızca destek vektörlerinin konumlarının önemli olmasıdır; marjdan uzakta ve doğru tarafta kalan noktalar uydurmayı değiştirmez. Teknik olarak bunun nedeni, bu noktaların modele uydurulurken kullanılan kayıp fonksiyonuna katkı vermemesidir.

Örneğin veri kümesinin ilk 60 ve ilk 120 noktasından öğrenilen modeli çizersek bunu görebiliriz (aşağıdaki şekil):


In [ ]:
# plot_svm_subsets.py
def plot_svm(N=10, ax=None):
    X, y = make_blobs(n_samples=200, centers=2,
                      random_state=0, cluster_std=0.60)
    X = X[:N]
    y = y[:N]
    model = SVC(kernel='linear', C=1E10)
    model.fit(X, y)
    
    ax = ax or plt.gca()
    ax.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='autumn')
    ax.set_xlim(-1, 4)
    ax.set_ylim(-1, 6)
    plot_svc_decision_function(model, ax)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
fig.subplots_adjust(left=0.0625, right=0.95, wspace=0.1)
for axi, N in zip(ax, [60, 120]):
    plot_svm(N, axi)
    axi.set_title('N = {0}'.format(N))



Sol panelde 60 eğitim noktası için model ve destek vektörlerini görürüz. Sağ panelde eğitim noktası sayısını ikiye katladık, ancak model değişmedi: soldaki üç destek vektörü sağdakilerle aynıdır. Uzak noktaların tam davranışına duyarsızlık, SVM modelinin güçlü yönlerinden biridir.

Bu not defterini canlı çalıştırıyorsanız, IPython etkileşimli araçlarıyla SVM modelinin bu özelliğini etkileşimli inceleyebilirsiniz:


In [ ]:
# interact_svm.py
from ipywidgets import interact, fixed
interact(plot_svm, N=(10, 200), ax=fixed(None));



### Doğrusal Sınırların Ötesinde: Çekirdek SVM


In [ ]:
# make_circles.py
from sklearn.datasets import make_circles
X, y = make_circles(100, factor=.1, noise=.1)

clf = SVC(kernel='linear').fit(X, y)

plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='autumn')
plot_svc_decision_function(clf, plot_support=False);



SVM'nin çok güçlü olabildiği yer, çekirdeklerle birleştirildiğinde ortaya çıkar. Çekirdekleri daha önce 5.6 Doğrusal Regresyon bölümündeki temel fonksiyon regresyonlarında gördük. Veriyi polinom ve Gauss temel fonksiyonlarıyla tanımlanan daha yüksek boyutlu bir uzaya yansıttık ve böylece doğrusal bir sınıflandırıcıyla doğrusal olmayan ilişkileri uyabildik.

SVM modellerinde aynı fikrin bir sürümünü kullanabiliriz. Çekirdeklere olan ihtiyacı motive etmek için doğrusal olarak ayrılamayan veriye bakalım (aşağıdaki şekil):


In [ ]:
# rbf_projection.py
r = np.exp(-(X ** 2).sum(1))



Hiçbir doğrusal ayırım bu veriyi asla ayıramayacaktır. Ancak 5.6 Doğrusal Regresyon bölümündeki temel fonksiyon regresyonlarından ders çıkarabiliriz: veriyi doğrusal bir ayırıcının yeterli olacağı daha yüksek bir boyuta nasıl yansıtabiliriz? Örneğin orta kümeye merkezlenmiş bir radyal temel fonksiyonu (RBF) hesaplayabiliriz:


In [ ]:
# rbf_3d_plot.py
from mpl_toolkits import mplot3d

ax = plt.subplot(projection='3d')
ax.scatter3D(X[:, 0], X[:, 1], r, c=y, s=50, cmap='autumn')
ax.view_init(elev=20, azim=30)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('r');



Bu ek veri boyutunu üç boyutlu bir çizimle görselleştirebiliriz (aşağıdaki şekil). Bu ek boyutla veri, örneğin r=0.7 düzleminde bir ayırıcı çizerek önemsiz biçimde doğrusal ayrılabilir hale gelir.


In [ ]:
# svc_rbf.py
clf = SVC(kernel='rbf', C=1E6)
clf.fit(X, y)



### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      Halka verisinde RBF çekirdekli SVM:
          
      from sklearn.datasets import make_circles
from sklearn.svm import SVC
X, y = make_circles(100, factor=0.1, noise=0.1)
clf = SVC(kernel='rbf', C=1e6)
clf.fit(X, y)
print("Doğruluk:", clf.score(X, y))

Daha önce tanımladığımız fonksiyonla uydurmayı görselleştirip destek vektörlerini belirleyelim (aşağıdaki şekil):


In [ ]:
# plot_svc_rbf.py
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='autumn')
plot_svc_decision_function(clf)
plt.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
            s=300, lw=1, facecolors='none');



Bu çekirdekli destek vektör makinesiyle uygun doğrusal olmayan karar sınırını öğreniriz. Bu çekirdek dönüşüm stratejisi, özellikle çekirdek hilekullanımının uygulanabildiği modellerde, hızlı doğrusal yöntemleri hızlı doğrusal olmayan yöntemlere dönüştürmek için sık kullanılır.

### SVM Ayarı: Marjı Yumuşatmak


In [ ]:
# soft_margin_data.py
X, y = make_blobs(n_samples=100, centers=2,
                  random_state=0, cluster_std=1.2)
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='autumn');



Tartışmamız şimdiye kadar mükemmel karar sınırının olduğu çok temiz veri kümelerine odaklandı. Verinizde bir miktar örtüşme varsa ne olur? Bu durumu ele almak için SVM uygulamasında marjı "yumuşatan" bir düzeltme faktörü vardır: daha iyi bir uyum için bazı noktaların marja girmesine izin verir. Marjın sertliği genelde C olarak bilinen bir ayar parametresiyle kontrol edilir.


In [ ]:
# soft_margin_C.py
X, y = make_blobs(n_samples=100, centers=2,
                  random_state=0, cluster_std=0.8)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
fig.subplots_adjust(left=0.0625, right=0.95, wspace=0.1)

for axi, C in zip(ax, [10.0, 0.1]):
    model = SVC(kernel='linear', C=C).fit(X, y)
    axi.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='autumn')
    plot_svc_decision_function(model, axi)
    axi.scatter(model.support_vectors_[:, 0],
                model.support_vectors_[:, 1],
                s=300, lw=1, facecolors='none');
    axi.set_title('C = {0:.1f}'.format(C), size=14)



Optimal C değeri veri kümenize bağlıdır; çapraz doğrulama veya benzeri bir yöntemle ayarlanmalıdır (bkz. 5.3 Hiperparametreler ve Model Doğrulama).

## Örnek: Yüz Tanıma


In [ ]:
# fetch_lfw_people.py
from sklearn.datasets import fetch_lfw_people
faces = fetch_lfw_people(min_faces_per_person=60)
print(faces.target_names)
print(faces.images.shape)



Destek vektör makinelerinin uygulaması olarak yüz tanıma problemini inceleyelim. Çeşitli kamuya mal olmuş kişilerin fotoğraflarından oluşan Labeled Faces in the Wild veri kümesini kullanacağız. Veri kümesi için bir getirici Scikit-Learn'e yerleştirilmiştir:


In [ ]:
# plot_faces.py
fig, ax = plt.subplots(3, 5, figsize=(8, 6))
for i, axi in enumerate(ax.flat):
    axi.imshow(faces.images[i], cmap='bone')
    axi.set(xticks=[], yticks=[],
            xlabel=faces.target_names[faces.target[i]])



Birkaç yüzü çizerek neyle çalıştığımıza bakalım (aşağıdaki şekil). Her görüntü 62×47, yaklaşık 3.000 piksel içerir. Her piksel değerini öznitelik olarak kullanabiliriz; ancak genelde daha anlamlı öznitelikler çıkaran bir ön işlemci daha etkilidir. Burada destek vektör makinesi sınıflandırıcısına beslemek için 150 temel bileşen çıkarmak üzere 5.9 PCA kullanacağız. Bunu ön işlemci ve sınıflandırıcıyı tek bir boru hattında paketleyerek yapabiliriz:


In [ ]:
# pca_svc_pipeline.py
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

pca = PCA(n_components=150, whiten=True,
          svd_solver='randomized', random_state=42)
svc = SVC(kernel='rbf', class_weight='balanced')
model = make_pipeline(pca, svc)



Sınıflandırıcı çıktısını test etmek için veriyi eğitim ve test kümelerine ayıracağız:


In [ ]:
# train_test_split_faces.py
from sklearn.model_selection import train_test_split
Xtrain, Xtest, ytrain, ytest = train_test_split(faces.data, faces.target,
                                                random_state=42)



Son olarak ızgara arama çapraz doğrulamasıyla parametre kombinasyonlarını keşfedebiliriz. Burada C (marj sertliği) ve gamma (RBF çekirdek boyutu) ayarlayıp en iyi modeli belirleyeceğiz:


In [ ]:
# gridsearch_svc.py
from sklearn.model_selection import GridSearchCV
param_grid = {'svc__C': [1, 5, 10, 50],
              'svc__gamma': [0.0001, 0.0005, 0.001, 0.005]}
grid = GridSearchCV(model, param_grid)

%time grid.fit(Xtrain, ytrain)
print(grid.best_params_)



Optimal değerler ızgaranın ortasına yakınsa iyi; kenarlardaysa gerçek optimumu bulmak için ızgarayı genişletmek isteyebilirsiniz. Bu çapraz doğrulanmış modelle henüz görülmemiş test verisinin etiketlerini tahmin edebiliriz:


In [ ]:
# predict_faces.py
model = grid.best_estimator_
yfit = model.predict(Xtest)



Küçük örneklemde birkaç test görüntüsünü tahminleriyle birlikte görelim (aşağıdaki şekil):


In [ ]:
# plot_test_predictions.py
fig, ax = plt.subplots(4, 6)
for i, axi in enumerate(ax.flat):
    axi.imshow(Xtest[i].reshape(62, 47), cmap='bone')
    axi.set(xticks=[], yticks=[])
    axi.set_ylabel(faces.target_names[yfit[i]].split()[-1],
                   color='black' if yfit[i] == ytest[i] else 'red')
fig.suptitle('Predicted Names; Incorrect Labels in Red', size=14);



Bu küçük örneklemde optimal tahmin edici yalnızca tek bir yüzü yanlış etiketledi. Sınıflandırma raporuyla etiket bazında performansa bakabiliriz:


In [ ]:
# classification_report_faces.py
from sklearn.metrics import classification_report
print(classification_report(ytest, yfit,
                            target_names=faces.target_names))



Sınıflar arası karışıklık matrisini de gösterebiliriz (aşağıdaki şekil). Bu, hangi etiketlerin birbiriyle karıştırılmaya eğilimli olduğuna dair fikir verir.


In [ ]:
# confusion_matrix_faces.py
from sklearn.metrics import confusion_matrix
import seaborn as sns
mat = confusion_matrix(ytest, yfit)
sns.heatmap(mat.T, square=True, annot=True, fmt='d',
            cbar=False, cmap='Blues',
            xticklabels=faces.target_names,
            yticklabels=faces.target_names)
plt.xlabel('true label')
plt.ylabel('predicted label');



Gerçek dünya yüz tanımada fotoğraflar düzgün ızgaralara önceden kırpılmamış olur; tek fark öznitelik seçimidir: yüzleri bulmak ve piksellemeden bağımsız öznitelikler çıkarmak için daha gelişmiş algoritmalar gerekir. Bu tür uygulamalar için OpenCV iyi bir seçenektir.

## Özet

Bu, destek vektör makinelerinin arkasındaki ilkelerin kısa sezgisel bir girişiydi.

Bu modeller güçlü bir sınıflandırma yöntemidir; birkaç nedenden dolayı:

Ancak SVM'lerin dezavantajları vardır:

Genelde SVM'lere, daha basit ve hızlı yöntemler yetersiz kaldığında geçerim. Yine de verinizde eğitim ve çapraz doğrulama için CPU döngüsü ayırabiliyorsanız, yöntem mükemmel sonuçlar verebilir.

> **Not**
>
